# Module 06 — Convolutional Neural Networks
## 06-02 Pooling & Receptive Fields

**Objective:** Implement MaxPool, AvgPool, and Global Average Pool from scratch,
derive the receptive-field growth formula, and compare pooling strategies against
stride-2 convolution on MNIST.

**Prerequisites:** 06-01 (Convolution from Scratch).


## Part 0 — Setup & Prerequisites

After each convolutional layer computes feature maps, a **pooling** layer reduces the
spatial dimensions.  This serves two purposes:
1. **Translation invariance** — a max-pool of 2×2 makes the network insensitive to
   small shifts of a feature within the pooling window.
2. **Computational efficiency** — halving the spatial size reduces FLOPs in subsequent
   layers by 4×.

In this notebook we:
1. Implement MaxPool2d, AvgPool2d, and Global Average Pool from scratch.
2. Visualise how each operation transforms a feature map.
3. Derive the **receptive field formula** and trace it through real CNN architectures.
4. Train four CNN variants (MaxPool / AvgPool / GAP / stride-2 conv) and compare.
5. Analyse what each pooling strategy preserves and discards.


In [ ]:
import sys
import math
import time
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

warnings.filterwarnings("ignore")

import random
print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy  : {np.__version__}")
if torch.cuda.is_available():
    print(f"CUDA   : {torch.version.cuda}")
    print(f"GPU    : {torch.cuda.get_device_name(0)}")


In [ ]:

SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
BATCH_SIZE    = 256
NUM_EPOCHS    = 5
LEARNING_RATE = 1e-3
NUM_CLASSES   = 10
DATA_ROOT     = "data/"

MNIST_MEAN = (0.1307,)
MNIST_STD  = (0.3081,)

print(f"Batch size    : {BATCH_SIZE}")
print(f"Epochs        : {NUM_EPOCHS}")
print(f"Learning rate : {LEARNING_RATE}")
print(f"Device        : {device}")


### Dataset: MNIST

We continue with MNIST so the visualisations from 06-01 remain directly comparable.
The single grey-scale channel means every feature map is easy to inspect at full resolution.


In [ ]:
# ── Load MNIST ────────────────────────────────────────────────────────────────
transform_mnist = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MNIST_MEAN, MNIST_STD),
])

full_train = torchvision.datasets.MNIST(
    root=DATA_ROOT, train=True, download=True, transform=transform_mnist
)
test_set = torchvision.datasets.MNIST(
    root=DATA_ROOT, train=False, download=True, transform=transform_mnist
)

generator  = torch.Generator().manual_seed(SEED)
train_size = int(0.9 * len(full_train))
val_size   = len(full_train) - train_size
train_set, val_set = random_split(full_train, [train_size, val_size], generator=generator)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=torch.cuda.is_available())
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=torch.cuda.is_available())
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=torch.cuda.is_available())

print(f"Train: {len(train_set):,}  |  Val: {len(val_set):,}  |  Test: {len(test_set):,}")
print(f"Image shape: {full_train[0][0].shape}")

# Load a batch of raw (un-normalised) images for pooling visualisations
raw_transform = transforms.ToTensor()
raw_mnist = torchvision.datasets.MNIST(root=DATA_ROOT, train=False,
                                        download=False, transform=raw_transform)
# Pick one sample image per digit class
class_imgs: dict[int, np.ndarray] = {}
for img_t, lbl in raw_mnist:
    if lbl not in class_imgs:
        class_imgs[lbl] = img_t.squeeze().numpy()
    if len(class_imgs) == 10:
        break

# Show the 10 representative digits
fig_d, axes_d = plt.subplots(1, 10, figsize=(15, 2))
for digit in range(10):
    axes_d[digit].imshow(class_imgs[digit], cmap="gray", vmin=0, vmax=1)
    axes_d[digit].set_title(str(digit), fontsize=10)
    axes_d[digit].axis("off")
plt.suptitle("One sample per MNIST digit class", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.show()

demo_img = class_imgs[7]   # digit '7' as running example throughout the notebook
print(f"Demo image (digit 7): shape={demo_img.shape}  "
      f"min={demo_img.min():.3f}  max={demo_img.max():.3f}")


## Part 1 — Pooling Operations from Scratch

### 1.1 MaxPool2d

Max-pooling slides a $K \times K$ window over the input and retains only the
**maximum** value in each window:

$$
Z_{h,w} = \max_{0 \le i < K,\; 0 \le j < K} X_{h \cdot s + i,\; w \cdot s + j}
$$

Intuition: if any part of a patch contains a strong positive response, the pooled
output will reflect it — regardless of the exact position within the window.
This grants **local translation invariance**.

### 1.2 AvgPool2d

Average-pooling replaces the max with a mean:

$$
Z_{h,w} = \frac{1}{K^2} \sum_{i=0}^{K-1}\sum_{j=0}^{K-1} X_{h \cdot s + i,\; w \cdot s + j}
$$

Intuition: averages smooth out local noise; the pooled value represents the
**mean activation energy** in the patch — better for gradients, less sparse.


In [ ]:
# ── MaxPool2d from scratch ────────────────────────────────────────────────────
def maxpool2d_scratch(
    x: np.ndarray,
    kernel_size: int = 2,
    stride: int | None = None,
) -> np.ndarray:
    '''Compute 2-D max-pooling using nested loops.

    Args:
        x: Input array of shape (H, W).
        kernel_size: Pooling window size (square).
        stride: Sliding step; defaults to kernel_size (non-overlapping).

    Returns:
        Pooled feature map of shape (H_out, W_out).
    '''
    if stride is None:
        stride = kernel_size
    H, W  = x.shape
    K     = kernel_size
    H_out = (H - K) // stride + 1
    W_out = (W - K) // stride + 1
    out   = np.zeros((H_out, W_out), dtype=np.float32)
    for h in range(H_out):
        for w in range(W_out):
            patch      = x[h * stride: h * stride + K, w * stride: w * stride + K]
            out[h, w]  = patch.max()
    return out


def avgpool2d_scratch(
    x: np.ndarray,
    kernel_size: int = 2,
    stride: int | None = None,
) -> np.ndarray:
    '''Compute 2-D average-pooling using nested loops.

    Args:
        x: Input array of shape (H, W).
        kernel_size: Pooling window size (square).
        stride: Sliding step; defaults to kernel_size (non-overlapping).

    Returns:
        Pooled feature map of shape (H_out, W_out).
    '''
    if stride is None:
        stride = kernel_size
    H, W  = x.shape
    K     = kernel_size
    H_out = (H - K) // stride + 1
    W_out = (W - K) // stride + 1
    out   = np.zeros((H_out, W_out), dtype=np.float32)
    for h in range(H_out):
        for w in range(W_out):
            patch      = x[h * stride: h * stride + K, w * stride: w * stride + K]
            out[h, w]  = patch.mean()
    return out


# ── Correctness check vs PyTorch ──────────────────────────────────────────────
x_test = torch.rand(1, 1, 8, 8)
x_np   = x_test[0, 0].numpy()

mp_scratch = maxpool2d_scratch(x_np, kernel_size=2, stride=2)
mp_ref     = F.max_pool2d(x_test, kernel_size=2, stride=2)[0, 0].numpy()
ap_scratch = avgpool2d_scratch(x_np, kernel_size=2, stride=2)
ap_ref     = F.avg_pool2d(x_test, kernel_size=2, stride=2)[0, 0].numpy()

print(f"MaxPool scratch vs F.max_pool2d: max error = {np.abs(mp_scratch - mp_ref).max():.2e}")
print(f"AvgPool scratch vs F.avg_pool2d: max error = {np.abs(ap_scratch - ap_ref).max():.2e}")
print(f"Input shape:  {x_np.shape}   Output shape: {mp_scratch.shape}")

# ── Timing: scratch vs PyTorch ─────────────────────────────────────────────────
REPS = 20
t0 = time.perf_counter()
for _ in range(REPS):
    maxpool2d_scratch(demo_img, kernel_size=2, stride=2)
t_scratch = (time.perf_counter() - t0) / REPS * 1000

x_demo_t = torch.tensor(demo_img).unsqueeze(0).unsqueeze(0)
t0 = time.perf_counter()
with torch.no_grad():
    for _ in range(REPS):
        F.max_pool2d(x_demo_t, kernel_size=2, stride=2)
t_pt = (time.perf_counter() - t0) / REPS * 1000

print(f"\n28x28 MaxPool(k=2, s=2) timing:")
print(f"  Scratch (nested loops) : {t_scratch:.3f} ms")
print(f"  PyTorch F.max_pool2d   : {t_pt:.3f} ms")
print(f"  Speedup                : {t_scratch / max(t_pt, 1e-9):.1f}x")


### 1.3 Visualising MaxPool vs AvgPool

Applying both operations to a raw MNIST digit makes their behaviour concrete:
- **MaxPool** keeps the brightest pixel in each window — output is sharper, higher values.
- **AvgPool** takes the mean — output is smoother, lower peak values, less sparse.


In [ ]:
# ── Visualise MaxPool vs AvgPool on MNIST digits ─────────────────────────────
pool_configs: list[dict] = [
    {"k": 2, "s": 2, "label": "k=2, s=2  (halve)"},
    {"k": 3, "s": 3, "label": "k=3, s=3  (third)"},
    {"k": 2, "s": 1, "label": "k=2, s=1  (overlap)"},
]

n_digits  = 3   # show digits 0, 4, 7
show_lbls = [0, 4, 7]
n_cols    = 1 + 2 * len(pool_configs)   # original + (max+avg) per config

fig_pool, axes_pool = plt.subplots(
    n_digits, n_cols, figsize=(2.5 * n_cols, 3 * n_digits)
)

for row_idx, digit in enumerate(show_lbls):
    img = class_imgs[digit]
    # Column 0: original
    axes_pool[row_idx][0].imshow(img, cmap="gray", vmin=0, vmax=1)
    axes_pool[row_idx][0].set_title(f"Digit {digit}\n28x28", fontsize=8)
    axes_pool[row_idx][0].axis("off")

    for cfg_idx, cfg in enumerate(pool_configs):
        mp_out = maxpool2d_scratch(img, kernel_size=cfg["k"], stride=cfg["s"])
        ap_out = avgpool2d_scratch(img, kernel_size=cfg["k"], stride=cfg["s"])

        col_mp = 1 + cfg_idx * 2
        col_ap = col_mp + 1

        axes_pool[row_idx][col_mp].imshow(mp_out, cmap="gray", vmin=0, vmax=1)
        axes_pool[row_idx][col_mp].set_title(
            f"MaxPool\n{cfg['label']}\n{mp_out.shape[0]}x{mp_out.shape[1]}", fontsize=7
        )
        axes_pool[row_idx][col_mp].axis("off")

        axes_pool[row_idx][col_ap].imshow(ap_out, cmap="gray", vmin=0, vmax=1)
        axes_pool[row_idx][col_ap].set_title(
            f"AvgPool\n{cfg['label']}\n{ap_out.shape[0]}x{ap_out.shape[1]}", fontsize=7
        )
        axes_pool[row_idx][col_ap].axis("off")

plt.suptitle("MaxPool vs AvgPool on MNIST Digits",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

# ── Pixel statistics comparison ───────────────────────────────────────────────
cfg0 = pool_configs[0]   # k=2, s=2
mp2 = maxpool2d_scratch(demo_img, kernel_size=cfg0["k"], stride=cfg0["s"])
ap2 = avgpool2d_scratch(demo_img, kernel_size=cfg0["k"], stride=cfg0["s"])

print(f"MaxPool(k=2, s=2) stats — shape: {mp2.shape}")
print(f"  mean={mp2.mean():.4f}  std={mp2.std():.4f}  max={mp2.max():.4f}  "
      f"sparsity={( mp2 == 0).mean():.2%}")
print(f"AvgPool(k=2, s=2) stats — shape: {ap2.shape}")
print(f"  mean={ap2.mean():.4f}  std={ap2.std():.4f}  max={ap2.max():.4f}  "
      f"sparsity={(ap2 == 0).mean():.2%}")
print(f"\nMaxPool preserves higher values (mean {mp2.mean():.3f} vs AvgPool {ap2.mean():.3f})")
print(f"AvgPool is smoother (std {ap2.std():.3f} vs MaxPool {mp2.std():.3f})")


### 1.3b Translation Invariance

A key property of MaxPool is **local translation invariance**: shifting an activation peak by up to $k-1$ pixels within a pooling window leaves the output unchanged.  We verify this empirically by measuring the Frobenius distance between the output for the original and shifted feature maps.


In [ ]:
# ── Translation invariance: MaxPool vs AvgPool vs raw activations ────────────
# MaxPool is maximally invariant within its window: shifting an activation peak
# by up to (k-1) pixels leaves the pool output unchanged.
# Here we measure this empirically by shifting a digit image by 0-3 pixels and
# comparing output similarity (Frobenius distance) for each strategy.

def apply_pool_numpy(
    img: np.ndarray,
    pool_fn: str,
    kernel_size: int = 2,
    stride: int = 2,
) -> np.ndarray:
    '''Apply a named pooling function to a single-channel image.

    Args:
        img: Input array of shape (H, W).
        pool_fn: One of "max", "avg", or "none".
        kernel_size: Pooling window size.
        stride: Pooling stride.

    Returns:
        Pooled array, or the original array if pool_fn="none".
    '''
    if pool_fn == "none":
        return img
    x_t  = torch.tensor(img).unsqueeze(0).unsqueeze(0)
    if pool_fn == "max":
        out = F.max_pool2d(x_t, kernel_size=kernel_size, stride=stride)
    else:
        out = F.avg_pool2d(x_t, kernel_size=kernel_size, stride=stride)
    return out[0, 0].numpy()


# Apply a Sobel-X kernel first to create a feature map
sobel_x_kn = np.array([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=np.float32)
base_feat   = apply_kern(demo_img, sobel_x_kn)   # (28, 28)

# Shift the base feature map by 0, 1, 2, 3 pixels (horizontal shift)
shifts = [0, 1, 2, 3]
pool_methods = [("none", "No Pool"), ("max", "MaxPool(k=2,s=2)"),
                ("avg", "AvgPool(k=2,s=2)")]

print("Frobenius distance from shift=0 (lower = more invariant):")
print(f"  {'Shift':>6s}  {'No Pool':>10s}  {'MaxPool':>10s}  {'AvgPool':>10s}")
print("-" * 46)

dist_records: list[dict] = []
for shift in shifts:
    shifted = np.roll(base_feat, shift, axis=1)   # circular shift
    row: dict = {"Shift": shift}
    for pool_fn, name in pool_methods:
        base_p   = apply_pool_numpy(base_feat, pool_fn)
        shifted_p = apply_pool_numpy(shifted,  pool_fn)
        dist      = np.linalg.norm(base_p - shifted_p, "fro")
        row[name] = dist
    dist_records.append(row)

for row in dist_records:
    print(f"  {row['Shift']:>6d}  {row['No Pool']:>10.3f}  "
          f"{row['MaxPool(k=2,s=2)']:>10.3f}  {row['AvgPool(k=2,s=2)']:>10.3f}")

# Plot the distance curves
fig_ti, ax_ti = plt.subplots(figsize=(7, 4))
for pool_fn, name in pool_methods:
    dists = [r[name] for r in dist_records]
    ax_ti.plot(shifts, dists, marker="o", lw=2, label=name)

ax_ti.set_xlabel("Horizontal shift (pixels)", fontsize=11)
ax_ti.set_ylabel("Frobenius distance from shift=0", fontsize=11)
ax_ti.set_title("Translation Invariance: MaxPool vs AvgPool vs No Pool",
                fontsize=11, fontweight="bold")
ax_ti.legend(fontsize=9)
ax_ti.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nConclusion: MaxPool shows the smallest distance increase with shift,")
print("confirming its stronger local translation invariance compared to AvgPool.")


### 1.4 Pool Window Size: Spatial Resolution vs Information Loss

A larger pool window discards more spatial information per step.  We quantify this by max-pooling with $k=2,3,4$ and measuring the reconstruction MSE after nearest-neighbour up-sampling back to $28 \times 28$.


In [ ]:
# ── Pool window size sweep: k=2,3,4 vs information retention ─────────────────
# Larger pooling windows down-sample more aggressively but may discard spatial info.
# We quantify information retention using the normalised SSIM-like measure:
# (1 - normalised MSE against up-sampled output) to keep it simple.

pool_window_sizes = [2, 3, 4]
digits_to_test    = [0, 1, 4, 7, 9]

print("Spatial down-sampling by pool window (applied to raw digit images):")
print(f"  {'Digit':>6s}  {'k=2 out':>10s}  {'k=3 out':>10s}  {'k=4 out':>10s}")
print(f"  {'':>6s}  {'MSE':>10s}  {'MSE':>10s}  {'MSE':>10s}")
print("-" * 48)

fig_ws, axes_ws = plt.subplots(len(digits_to_test), len(pool_window_sizes) + 1,
                                figsize=(3 * (len(pool_window_sizes) + 1),
                                         2.5 * len(digits_to_test)))

for row_i, digit in enumerate(digits_to_test):
    img = class_imgs[digit]
    # Column 0: original
    axes_ws[row_i][0].imshow(img, cmap="gray", vmin=0, vmax=1)
    axes_ws[row_i][0].set_title(f"Digit {digit}\n28x28", fontsize=8)
    axes_ws[row_i][0].axis("off")

    mses: list[float] = []
    for col_j, k in enumerate(pool_window_sizes, start=1):
        img_t   = torch.tensor(img).unsqueeze(0).unsqueeze(0)
        pooled  = F.max_pool2d(img_t, kernel_size=k, stride=k)[0, 0].numpy()
        # Up-sample back using nearest-neighbour to compute MSE
        upsampled = np.kron(pooled, np.ones((k, k)))[:28, :28]
        mse_val   = float(np.mean((img - upsampled) ** 2))
        mses.append(mse_val)

        axes_ws[row_i][col_j].imshow(pooled, cmap="gray", vmin=0, vmax=1)
        axes_ws[row_i][col_j].set_title(
            f"k={k}\n{pooled.shape[0]}x{pooled.shape[1]}\nMSE={mse_val:.4f}", fontsize=7
        )
        axes_ws[row_i][col_j].axis("off")

    print(f"  {digit:>6d}  {mses[0]:>10.4f}  {mses[1]:>10.4f}  {mses[2]:>10.4f}")

plt.suptitle("MaxPool Window Size Sweep — Spatial Resolution vs Information Loss",
             fontsize=10, fontweight="bold")
plt.tight_layout()
plt.show()

# Average MSE across digits
all_mses = {k: [] for k in pool_window_sizes}
for digit in range(10):
    img   = class_imgs[digit]
    img_t = torch.tensor(img).unsqueeze(0).unsqueeze(0)
    for k in pool_window_sizes:
        pooled    = F.max_pool2d(img_t, kernel_size=k, stride=k)[0, 0].numpy()
        upsampled = np.kron(pooled, np.ones((k, k)))[:28, :28]
        all_mses[k].append(float(np.mean((img - upsampled) ** 2)))

print("\nAverage reconstruction MSE (lower = more info preserved):")
for k in pool_window_sizes:
    avg_mse = np.mean(all_mses[k])
    print(f"  MaxPool(k={k})  avg MSE = {avg_mse:.4f}  "
          f"output size = {28//k}x{28//k}")


## Part 2 — Global Average Pool & Receptive Fields

### 2.1 Global Average Pool (GAP)

Global Average Pooling collapses the **entire spatial dimension** to a single value
per channel:

$$
Z_c = \frac{1}{H \cdot W} \sum_{h=0}^{H-1}\sum_{w=0}^{W-1} X_{c,h,w}
$$

GAP produces a vector of length $C$ (one scalar per feature map).  This replaces the
large fully-connected head in modern architectures (e.g., ResNet, MobileNet): instead
of $C \cdot H \cdot W \to 512 \to \text{classes}$, we get $C \to \text{classes}$.

**Benefits:**
- Eliminates the parameter-heavy FC layers.
- The network can accept **any input spatial size** — no flattening needed.
- Acts as a structural regulariser (forces each channel to specialise).

### 2.2 Receptive Field

The **receptive field** of a neuron at layer $l$ is the region in the original input
that can influence its value.  For stacked conv/pool layers:

$$
\text{RF}_{l} = \text{RF}_{l-1} + (k_l - 1) \cdot j_{l-1}
\qquad
j_l = j_{l-1} \cdot s_l
$$

where $j_l$ is the **jump** (how many input pixels one output pixel corresponds to after
$l$ layers) and $s_l$ is the stride of layer $l$.  Starting values: $\text{RF}_0 = 1$,
$j_0 = 1$.


In [ ]:
# ── Global Average Pool from scratch ─────────────────────────────────────────
def global_avg_pool(x: torch.Tensor) -> torch.Tensor:
    '''Compute Global Average Pooling — collapse spatial dims to scalars.

    Args:
        x: Feature map tensor of shape (N, C, H, W).

    Returns:
        Pooled tensor of shape (N, C, 1, 1) — one scalar per channel per sample.
    '''
    return x.mean(dim=(-2, -1), keepdim=True)


# Verify against nn.AdaptiveAvgPool2d(1)
x_gap_test  = torch.randn(4, 16, 7, 7)
gap_scratch = global_avg_pool(x_gap_test)
gap_ref     = nn.AdaptiveAvgPool2d(1)(x_gap_test)

max_err_gap = (gap_scratch - gap_ref).abs().max().item()
print(f"global_avg_pool vs AdaptiveAvgPool2d(1): max error = {max_err_gap:.2e}  [OK]")
print(f"Input:  {x_gap_test.shape}")
print(f"Output: {gap_scratch.shape}  -> squeeze to {gap_scratch.squeeze(-1).squeeze(-1).shape}")

# ── Visualise what GAP does ───────────────────────────────────────────────────
# Simulate 4 channels (hand-crafted feature maps of the demo digit)
sobel_x   = np.array([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=np.float32)
sobel_y   = sobel_x.T
blur_k    = np.array([[1,2,1],[2,4,2],[1,2,1]], dtype=np.float32) / 16.0
laplacian = np.array([[0,1,0],[1,-4,1],[0,1,0]], dtype=np.float32)
kernels_gap = [sobel_x, sobel_y, blur_k, laplacian]
names_gap   = ["Sobel-X", "Sobel-Y", "Gaussian", "Laplacian"]

def apply_kern(img: np.ndarray, kern: np.ndarray) -> np.ndarray:
    '''Apply a small kernel to an image via zero-padded cross-correlation.

    Pure NumPy implementation — no scipy required.

    Args:
        img: Input array of shape (H, W), values in [0, 1].
        kern: Filter array of shape (kH, kW).

    Returns:
        Feature map of shape (H, W) — same spatial size as input.
    '''
    H, W   = img.shape
    kH, kW = kern.shape
    pad    = kH // 2
    img_p  = np.pad(img, pad, mode="constant", constant_values=0.0)
    out    = np.zeros((H, W), dtype=np.float32)
    for h in range(H):
        for w in range(W):
            patch      = img_p[h: h + kH, w: w + kW]
            out[h, w]  = float(np.sum(patch * kern))
    return out

feat_maps_gap = np.stack([apply_kern(demo_img, k) for k in kernels_gap])  # (4, 28, 28)
gap_values    = feat_maps_gap.mean(axis=(1, 2))   # (4,) — one scalar per channel

fig_gap, axes_gap = plt.subplots(1, len(kernels_gap) + 1, figsize=(14, 3.5))
for k_idx, (name, fmap) in enumerate(zip(names_gap, feat_maps_gap)):
    v = np.abs(fmap).max() or 1.0
    axes_gap[k_idx].imshow(fmap, cmap="RdBu_r", vmin=-v, vmax=v)
    axes_gap[k_idx].set_title(f"{name}\nGAP={gap_values[k_idx]:.3f}", fontsize=9)
    axes_gap[k_idx].axis("off")

# Show GAP as a bar chart in the last column
axes_gap[-1].bar(range(len(names_gap)), gap_values,
                  color=["#1f77b4","#ff7f0e","#2ca02c","#d62728"])
axes_gap[-1].set_xticks(range(len(names_gap)))
axes_gap[-1].set_xticklabels(names_gap, rotation=20, fontsize=8)
axes_gap[-1].set_ylabel("GAP value", fontsize=9)
axes_gap[-1].set_title("Global Average\nPooling result", fontsize=9)
axes_gap[-1].axhline(0, color="k", lw=0.8, ls="--")
axes_gap[-1].grid(axis="y", alpha=0.3)

plt.suptitle(f"Global Average Pooling: Each Feature Map -> One Scalar  (digit '7')",
             fontsize=10, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nGAP values per channel:")
for name, val in zip(names_gap, gap_values):
    print(f"  {name:<12s}: {val:+.4f}")


# ── GAP parameter savings vs FC head ─────────────────────────────────────────
# Replacing a large FC head with GAP saves parameters proportional to spatial size.
print("\nGAP vs FC head parameter savings (C=64 channels):")
print(f"  {'Spatial size':>14s}  {'FC params (64xHxW->256)':>25s}  {'GAP saves':>12s}")
for h_w in [28, 14, 7, 4, 1]:
    fc_params  = 64 * h_w * h_w * 256 + 256
    gap_params = 64 * 256 + 256          # after GAP: 64->256
    savings    = fc_params - gap_params
    print(f"  {h_w:>6d}x{h_w:<6d}  {fc_params:>25,d}  {savings:>12,d}")
print("  GAP collapses H x W to 1x1, eliminating H*W factor from FC input.")


### 2.3 Computing the Receptive Field

The receptive field formula lets us predict, without running the network, which input
region each output neuron "sees".  This is critical for architecture design:
- Too small an RF → neurons lack context (can't distinguish global shapes).
- Too large an RF → over-aggregation, spatial information lost too early.

Modern networks balance RF growth through careful kernel size / stride choices.


In [ ]:
# ── Receptive Field formula ───────────────────────────────────────────────────
def compute_receptive_field(
    layer_configs: list[tuple[int, int]],
) -> list[dict]:
    '''Compute the receptive field at each layer of a CNN.

    Uses the recurrence:
        RF[l]   = RF[l-1]   + (k[l] - 1) * jump[l-1]
        jump[l] = jump[l-1] * stride[l]

    Starting values: RF[0] = 1, jump[0] = 1.

    Args:
        layer_configs: List of (kernel_size, stride) tuples, one per layer.
            Use kernel_size=2, stride=2 for MaxPool2d(2).

    Returns:
        List of dicts with keys: layer_idx, kernel, stride, RF, jump.
    '''
    rf   = 1
    jump = 1
    rows: list[dict] = [{"Layer": 0, "kernel": "-", "stride": "-",
                          "RF": rf, "jump": jump, "note": "input"}]
    for idx, (k, s) in enumerate(layer_configs, start=1):
        rf   = rf + (k - 1) * jump
        jump = jump * s
        rows.append({"Layer": idx, "kernel": k, "stride": s,
                      "RF": rf, "jump": jump, "note": ""})
    return rows


# ── Architecture 1: SimpleCNN from 06-01 (MaxPool variant) ───────────────────
#   Conv(k=3,s=1) -> MaxPool(k=2,s=2) -> Conv(k=3,s=1) -> MaxPool(k=2,s=2)
#   -> Conv(k=3,s=1) -> AdaptiveAvgPool(4)
simple_cnn_layers = [(3,1),(2,2),(3,1),(2,2),(3,1)]
rf_simple = compute_receptive_field(simple_cnn_layers)
print("SimpleCNN (from 06-01):")
for row in rf_simple:
    k_s  = f"k={row['kernel']}, s={row['stride']}" if row["note"] != "input" else "input"
    print(f"  Layer {row['Layer']:>2d}  {k_s:<18s}  RF={row['RF']:>4d}  jump={row['jump']:>3d}")

# ── Architecture 2: All stride-2 convolutions (no explicit pooling) ───────────
#   Conv(k=3,s=1) -> Conv(k=3,s=2) -> Conv(k=3,s=1) -> Conv(k=3,s=2) -> Conv(k=3,s=1)
stride2_layers = [(3,1),(3,2),(3,1),(3,2),(3,1)]
rf_stride2 = compute_receptive_field(stride2_layers)
print("\nStride-2 Conv variant:")
for row in rf_stride2:
    k_s  = f"k={row['kernel']}, s={row['stride']}" if row["note"] != "input" else "input"
    print(f"  Layer {row['Layer']:>2d}  {k_s:<18s}  RF={row['RF']:>4d}  jump={row['jump']:>3d}")

# ── Architecture 3: VGG-style (more conv before each pool) ───────────────────
#   2x Conv(k=3,s=1) -> MaxPool(k=2,s=2) -> 2x Conv(k=3,s=1) -> MaxPool(k=2,s=2)
vgg_style_layers = [(3,1),(3,1),(2,2),(3,1),(3,1),(2,2)]
rf_vgg = compute_receptive_field(vgg_style_layers)
print("\nVGG-style (2x conv per pool block):")
for row in rf_vgg:
    k_s = f"k={row['kernel']}, s={row['stride']}" if row["note"] != "input" else "input"
    print(f"  Layer {row['Layer']:>2d}  {k_s:<18s}  RF={row['RF']:>4d}  jump={row['jump']:>3d}")

# ── Visualise RF growth ───────────────────────────────────────────────────────
fig_rf, ax_rf = plt.subplots(figsize=(9, 4))
for label, rf_data, color in [
    ("SimpleCNN (MaxPool)", rf_simple, "#1f77b4"),
    ("Stride-2 Conv",       rf_stride2, "#ff7f0e"),
    ("VGG-style",           rf_vgg,    "#2ca02c"),
]:
    layers = [r["Layer"] for r in rf_data]
    rfs    = [r["RF"]    for r in rf_data]
    ax_rf.plot(layers, rfs, marker="o", lw=2, label=label, color=color)

ax_rf.axhline(28, color="k", ls="--", lw=1, label="Input size (28px)")
ax_rf.set_xlabel("Layer index", fontsize=11)
ax_rf.set_ylabel("Receptive Field (pixels)", fontsize=11)
ax_rf.set_title("Receptive Field Growth Through CNN Layers", fontsize=11, fontweight="bold")
ax_rf.legend(fontsize=9)
ax_rf.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ── RF comparison table ───────────────────────────────────────────────────────
print("\nFinal RF after all layers (covers 28x28 input = full image):")
print(f"  {'Architecture':<35s}  {'Final RF':>9s}  {'Jumps/stride':>13s}")
for label, rf_data in [("SimpleCNN (MaxPool)", rf_simple),
                        ("Stride-2 Conv",       rf_stride2),
                        ("VGG-style",           rf_vgg)]:
    final = rf_data[-1]
    covers_full = "YES" if final["RF"] >= 28 else "no"
    print(f"  {label:<35s}  {final['RF']:>9d}  {final['jump']:>13d}  ({covers_full})")


# ── Theoretical RF required to see the full 28x28 MNIST image ────────────────
# How many conv+pool layers (k=3, s=1 conv + k=2, s=2 pool) does it take for
# the receptive field to cover the full 28x28 input?
print("\nHow many conv+pool blocks to cover 28x28 input?")
print(f"  {'Blocks':>7s}  {'RF after':>9s}  {'Covers?':>8s}")
running_layers: list[tuple[int, int]] = []
for n_blocks in range(1, 7):
    running_layers.append((3, 1))   # conv k=3, s=1
    running_layers.append((2, 2))   # maxpool k=2, s=2
    rf_data = compute_receptive_field(running_layers)
    final_rf = rf_data[-1]["RF"]
    covers   = "YES" if final_rf >= 28 else "no"
    print(f"  {n_blocks:>7d}  {final_rf:>9d}px  {covers:>8s}")
    if final_rf >= 28:
        print(f"  => Full 28x28 coverage reached after {n_blocks} conv+pool block(s)")
        break

# ── RF at each spatial position for SimpleCNN ──────────────────────────────────
# For a 28x28 input, each output neuron of the SimpleCNN (after 5 layers)
# corresponds to a specific region in the input.
# Compute the actual spatial coverage: RF spans from centre-RF//2 to centre+RF//2.
final_rf_val = rf_simple[-1]["RF"]
final_jump   = rf_simple[-1]["jump"]

# For a 4x4 final feature map (SimpleCNN after AdaptiveAvgPool(4)):
# The centre of each output cell, in input coordinates, is spaced 'jump' apart.
out_size = 4   # after AdaptiveAvgPool(4)
print(f"\nSimpleCNN: 4x4 output, RF={final_rf_val}px, jump={final_jump}px")
print(f"  Each of the {out_size*out_size} output neurons covers a ~{final_rf_val}x{final_rf_val} region.")
print(f"  Adjacent neurons are spaced {final_jump}px apart in the 28x28 input.")
overlap = max(0, final_rf_val - final_jump)
print(f"  Receptive field overlap between adjacent neurons: {overlap}px")


## Part 3 — Comparing Pooling Strategies

We train **four CNN variants** with identical architectures except for the spatial
down-sampling mechanism:

| Variant | Down-sampling | Notes |
|---------|---------------|-------|
| **MaxPool** | `nn.MaxPool2d(2)` | Standard; good for sparse/sharp features |
| **AvgPool** | `nn.AvgPool2d(2)` | Smoother; better gradient flow |
| **GAP** | `nn.AdaptiveAvgPool2d(1)` | Collapses all spatial to 1×1 before FC |
| **Stride-2 Conv** | Conv(k=3, s=2) | Learnable down-sampling; most flexible |

All variants share: 3 conv layers (1→16→32→64 channels), same FC head, Adam optimiser,
5 epochs on MNIST.


In [ ]:
class PoolCNN(nn.Module):
    '''CNN with configurable spatial down-sampling strategy.

    Supports MaxPool, AvgPool, Global Average Pool, and stride-2 convolution.
    The feature extractor produces a (N, 64, H_out, W_out) tensor; the
    classifier head is a two-layer MLP.

    Attributes:
        pool_type: Identifier string for the pooling strategy.
        features: Sequential convolutional feature extractor.
        classifier: Sequential fully-connected classifier head.
    '''

    SUPPORTED_TYPES = ("maxpool", "avgpool", "gap", "stride2")

    def __init__(
        self,
        pool_type: str = "maxpool",
        num_classes: int = 10,
        dropout_rate: float = 0.3,
    ) -> None:
        '''Build PoolCNN with the specified spatial down-sampling strategy.

        Args:
            pool_type: One of "maxpool", "avgpool", "gap", or "stride2".
            num_classes: Number of output classes.
            dropout_rate: Dropout probability in the classifier head.
        '''
        super().__init__()
        assert pool_type in self.SUPPORTED_TYPES, (
            f"pool_type must be one of {self.SUPPORTED_TYPES}"
        )
        self.pool_type = pool_type

        if pool_type in ("maxpool", "avgpool", "gap"):
            # Standard conv layers + pooling layer
            pool_layer1 = (nn.MaxPool2d(2) if pool_type == "maxpool"
                           else nn.AvgPool2d(2))
            pool_layer2 = (nn.MaxPool2d(2) if pool_type == "maxpool"
                           else nn.AvgPool2d(2))

            if pool_type == "gap":
                # No intermediate pooling — use GAP at the end
                self.features = nn.Sequential(
                    nn.Conv2d(1, 16, kernel_size=3, padding=1),   # 28x28
                    nn.ReLU(inplace=True),
                    nn.Conv2d(16, 32, kernel_size=3, padding=1),  # 28x28
                    nn.ReLU(inplace=True),
                    nn.Conv2d(32, 64, kernel_size=3, padding=1),  # 28x28
                    nn.ReLU(inplace=True),
                    nn.AdaptiveAvgPool2d(1),                       # 1x1
                )
                fc_in = 64
            else:
                self.features = nn.Sequential(
                    nn.Conv2d(1, 16, kernel_size=3, padding=1),
                    nn.ReLU(inplace=True),
                    pool_layer1,                                   # 14x14

                    nn.Conv2d(16, 32, kernel_size=3, padding=1),
                    nn.ReLU(inplace=True),
                    pool_layer2,                                   # 7x7

                    nn.Conv2d(32, 64, kernel_size=3, padding=1),
                    nn.ReLU(inplace=True),
                    nn.AdaptiveAvgPool2d(4),                       # 4x4
                )
                fc_in = 64 * 4 * 4
        else:
            # stride2: stride-2 conv replaces pooling layers
            self.features = nn.Sequential(
                nn.Conv2d(1, 16, kernel_size=3, padding=1, stride=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(16, 16, kernel_size=3, padding=1, stride=2),  # 14x14

                nn.Conv2d(16, 32, kernel_size=3, padding=1, stride=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(32, 32, kernel_size=3, padding=1, stride=2),  # 7x7

                nn.Conv2d(32, 64, kernel_size=3, padding=1, stride=1),
                nn.ReLU(inplace=True),
                nn.AdaptiveAvgPool2d(4),                                 # 4x4
            )
            fc_in = 64 * 4 * 4

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(fc_in, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''Run the forward pass.

        Args:
            x: Input images of shape (N, 1, 28, 28).

        Returns:
            Class logits of shape (N, num_classes).
        '''
        return self.classifier(self.features(x))


def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> tuple[float, float]:
    '''Train the model for one epoch.

    Args:
        model: The CNN model.
        dataloader: Training DataLoader.
        criterion: Loss function.
        optimizer: Optimizer instance.
        device: Compute device.

    Returns:
        Tuple of (average_loss, accuracy).
    '''
    model.train()
    running_loss = 0.0
    correct = 0
    total   = 0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct      += logits.argmax(1).eq(labels).sum().item()
        total        += labels.size(0)
    return running_loss / total, correct / total


def evaluate(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    '''Evaluate the model on a dataset split.

    Args:
        model: The CNN model.
        dataloader: Validation or test DataLoader.
        criterion: Loss function.
        device: Compute device.

    Returns:
        Tuple of (average_loss, accuracy).
    '''
    model.eval()
    running_loss = 0.0
    correct = 0
    total   = 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss   = criterion(logits, labels)
            running_loss += loss.item() * images.size(0)
            correct      += logits.argmax(1).eq(labels).sum().item()
            total        += labels.size(0)
    return running_loss / total, correct / total


# ── Instantiate & inspect parameter counts ────────────────────────────────────
criterion = nn.CrossEntropyLoss()
pool_variants: list[str] = ["maxpool", "avgpool", "gap", "stride2"]

print(f"{'Variant':<12s}  {'Params':>10s}  {'Feature out':>12s}")
print("-" * 40)
for ptype in pool_variants:
    m = PoolCNN(pool_type=ptype)
    n_p = sum(p.numel() for p in m.parameters())
    with torch.no_grad():
        feat_out = m.features(torch.zeros(1, 1, 28, 28))
    print(f"  {ptype:<10s}  {n_p:>10,d}  {str(tuple(feat_out.shape)):>12s}")


In [ ]:
# ── Train all four variants ───────────────────────────────────────────────────
histories: dict[str, dict] = {}

for pool_type in pool_variants:
    print(f"\n{'='*55}")
    print(f"Training variant: {pool_type.upper()}")
    print(f"{'='*55}")

    torch.manual_seed(SEED)
    model_v   = PoolCNN(pool_type=pool_type, num_classes=NUM_CLASSES).to(device)
    optimizer = torch.optim.Adam(model_v.parameters(), lr=LEARNING_RATE)

    tr_losses, vl_losses, tr_accs, vl_accs = [], [], [], []
    best_val  = float("inf")
    best_wts  = None
    t_total   = 0.0

    for epoch in range(NUM_EPOCHS):
        t0  = time.time()
        tl, ta = train_one_epoch(model_v, train_loader, criterion, optimizer, device)
        vl, va = evaluate(model_v, val_loader, criterion, device)
        elapsed = time.time() - t0
        t_total += elapsed

        tr_losses.append(tl); vl_losses.append(vl)
        tr_accs.append(ta);   vl_accs.append(va)

        if vl < best_val:
            best_val = vl
            best_wts = {k: v.clone() for k, v in model_v.state_dict().items()}

        print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
              f"Train Loss: {tl:.4f} | Val Loss: {vl:.4f} | "
              f"Val Acc: {va:.2%} | Time: {elapsed:.1f}s")

    model_v.load_state_dict(best_wts)
    test_loss, test_acc = evaluate(model_v, test_loader, criterion, device)

    histories[pool_type] = {
        "model":      model_v,
        "train_loss": tr_losses,
        "val_loss":   vl_losses,
        "train_acc":  tr_accs,
        "val_acc":    vl_accs,
        "test_acc":   test_acc,
        "test_loss":  test_loss,
        "total_time": t_total,
    }
    print(f"Test Accuracy: {test_acc:.2%}")


In [ ]:
# ── Training curves: all four variants on one plot ────────────────────────────
colors   = {"maxpool": "#1f77b4", "avgpool": "#ff7f0e",
            "gap":     "#2ca02c", "stride2": "#d62728"}
styles   = {"maxpool": "-",  "avgpool": "--", "gap": "-.", "stride2": ":"}
epochs_x = range(1, NUM_EPOCHS + 1)

fig_tr, axes_tr = plt.subplots(1, 2, figsize=(13, 4))

for ptype in pool_variants:
    h   = histories[ptype]
    col = colors[ptype]
    sty = styles[ptype]
    axes_tr[0].plot(epochs_x, h["val_loss"], color=col, ls=sty, lw=2,
                    label=ptype, marker="o", markersize=4)
    axes_tr[1].plot(epochs_x, [a*100 for a in h["val_acc"]], color=col, ls=sty,
                    lw=2, label=ptype, marker="o", markersize=4)

axes_tr[0].set_xlabel("Epoch"); axes_tr[0].set_ylabel("Val Loss")
axes_tr[0].set_title("Validation Loss"); axes_tr[0].legend(fontsize=9)
axes_tr[0].grid(alpha=0.3)

axes_tr[1].set_xlabel("Epoch"); axes_tr[1].set_ylabel("Val Accuracy (%)")
axes_tr[1].set_title("Validation Accuracy"); axes_tr[1].legend(fontsize=9)
axes_tr[1].grid(alpha=0.3)

plt.suptitle("Pooling Strategy Comparison — MNIST Validation Curves",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()


## Part 4 — Evaluation & Analysis

### 4.1 Test Accuracy Summary

We compare the four pooling strategies on the official 10 K test set and analyse
parameter count, training time, and accuracy.


In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
summary_rows: list[dict] = []
for ptype in pool_variants:
    h = histories[ptype]
    m = h["model"]
    n_p = sum(p.numel() for p in m.parameters())
    summary_rows.append({
        "Variant":       ptype,
        "Params":        n_p,
        "Test Acc":      f"{h['test_acc']:.4f}",
        "Val Acc (ep5)": f"{h['val_acc'][-1]:.4f}",
        "Train Time (s)":f"{h['total_time']:.1f}",
    })

summary_df = pd.DataFrame(summary_rows)
print("Pooling strategy comparison on MNIST:")
print(summary_df.to_string(index=False))

# Bar chart of test accuracy
fig_bar, ax_bar = plt.subplots(figsize=(7, 4))
labels_bar = [r["Variant"] for r in summary_rows]
accs_bar   = [histories[r["Variant"]]["test_acc"] * 100 for r in summary_rows]
bar_colors = [colors[r["Variant"]] for r in summary_rows]

bars = ax_bar.bar(range(len(labels_bar)), accs_bar, color=bar_colors,
                   edgecolor="k", lw=0.7)
for bar, acc in zip(bars, accs_bar):
    ax_bar.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                f"{acc:.2f}%", ha="center", va="bottom", fontsize=9)

ax_bar.set_xticks(range(len(labels_bar)))
ax_bar.set_xticklabels(labels_bar, fontsize=10)
ax_bar.set_ylabel("Test Accuracy (%)", fontsize=11)
ax_bar.set_title("Test Accuracy by Pooling Strategy", fontsize=11, fontweight="bold")
ax_bar.set_ylim(min(accs_bar) - 1, max(accs_bar) + 1)
ax_bar.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

best_variant = max(pool_variants, key=lambda p: histories[p]["test_acc"])
print(f"\nBest variant: {best_variant} ({histories[best_variant]['test_acc']:.4f})")


# ── Per-variant confusion matrices ────────────────────────────────────────────
fig_cms, axes_cms = plt.subplots(1, len(pool_variants), figsize=(20, 5))
for ax, ptype in zip(axes_cms, pool_variants):
    m_v = histories[ptype]["model"]
    all_p: list[int] = []
    all_l: list[int] = []
    m_v.eval()
    with torch.no_grad():
        for imgs, lbls in test_loader:
            logits = m_v(imgs.to(device))
            all_p.extend(logits.argmax(1).cpu().tolist())
            all_l.extend(lbls.tolist())
    cm_v = confusion_matrix(all_l, all_p)
    disp = ConfusionMatrixDisplay(cm_v, display_labels=list(range(10)))
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    test_a = histories[ptype]["test_acc"]
    ax.set_title(f"{ptype}\n({test_a:.2%})", fontsize=9, fontweight="bold")

plt.suptitle("Confusion Matrices — Four Pooling Strategies", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

# ── Per-class accuracy for each variant ────────────────────────────────────────
print("\nPer-class accuracy comparison:")
print(f"  {'Digit':>5s}  " + "  ".join(f"{p:>10s}" for p in pool_variants))
print("-" * (7 + 12 * len(pool_variants)))
for digit in range(10):
    row_parts = [f"  {digit:>5d}"]
    for ptype in pool_variants:
        m_v  = histories[ptype]["model"]
        all_p_d: list[int] = []
        all_l_d: list[int] = []
        m_v.eval()
        with torch.no_grad():
            for imgs, lbls in test_loader:
                logits = m_v(imgs.to(device))
                all_p_d.extend(logits.argmax(1).cpu().tolist())
                all_l_d.extend(lbls.tolist())
        mask    = [l == digit for l in all_l_d]
        n_c     = sum(p == l for p, l, m in zip(all_p_d, all_l_d, mask) if m)
        n_total = sum(mask)
        row_parts.append(f"  {n_c/max(n_total,1):>9.2%}")
    print("".join(row_parts))


### 4.2 Pooling Effect on Feature Maps

We extract the first-layer feature maps before and after the first MaxPool/AvgPool
operation to visualise how pooling changes the spatial structure of activations.


In [ ]:
# ── Feature maps before and after MaxPool vs AvgPool ─────────────────────────
sample_tensor = test_set[0][0].unsqueeze(0).to(device)   # (1, 1, 28, 28)
sample_digit  = int(test_set[0][1])

def get_pool_maps(
    pool_model: PoolCNN,
    img: torch.Tensor,
    pool_device: torch.device,
) -> tuple[torch.Tensor, torch.Tensor]:
    '''Extract feature maps before and after the first pooling operation.

    Args:
        pool_model: A PoolCNN instance.
        img: Single image of shape (1, 1, 28, 28).
        pool_device: Device to run the model on.

    Returns:
        Tuple of (pre_pool, post_pool) tensors with shape (1, C, H, W).
    '''
    pool_model.eval()
    x = img.to(pool_device)
    with torch.no_grad():
        # Walk through features layer by layer
        pre_pool  = None
        post_pool = None
        feat_seq  = pool_model.features
        hit_pool  = False
        for layer in feat_seq:
            x = layer(x)
            if isinstance(layer, nn.ReLU) and not hit_pool:
                pre_pool = x.clone()
            if isinstance(layer, (nn.MaxPool2d, nn.AvgPool2d)) and not hit_pool:
                post_pool = x.clone()
                hit_pool  = True
    return pre_pool, post_pool


# Compare MaxPool vs AvgPool feature maps
mp_model  = histories["maxpool"]["model"]
ap_model  = histories["avgpool"]["model"]

mp_pre,  mp_post  = get_pool_maps(mp_model, sample_tensor, device)
ap_pre,  ap_post  = get_pool_maps(ap_model, sample_tensor, device)

# Show 8 channels per model
n_show_ch = 8
fig_pfm, axes_pfm = plt.subplots(4, n_show_ch, figsize=(2.2 * n_show_ch, 9))
fig_pfm.suptitle(
    f"Feature Maps — Digit '{sample_digit}'\n"
    "Rows 0-1: MaxPool (pre / post)   Rows 2-3: AvgPool (pre / post)",
    fontsize=10, fontweight="bold"
)
for col_k in range(n_show_ch):
    for row_base, pre, post in [(0, mp_pre, mp_post), (2, ap_pre, ap_post)]:
        for row_off, maps in enumerate([pre, post]):
            feat = maps[0, col_k].cpu().numpy()
            v    = np.abs(feat).max() or 1.0
            axes_pfm[row_base + row_off][col_k].imshow(
                feat, cmap="RdBu_r", vmin=-v, vmax=v
            )
            axes_pfm[row_base + row_off][col_k].set_title(f"ch{col_k}", fontsize=7)
            axes_pfm[row_base + row_off][col_k].axis("off")

plt.tight_layout()
plt.show()

# Activation stats: pre vs post pool
for label, pre, post in [("MaxPool", mp_pre, mp_post), ("AvgPool", ap_pre, ap_post)]:
    pre_np  = pre.cpu().numpy()
    post_np = post.cpu().numpy()
    print(f"{label}:")
    print(f"  Pre-pool   shape={pre_np.shape}  mean={pre_np.mean():.4f}  "
          f"std={pre_np.std():.4f}  sparsity={(pre_np==0).mean():.2%}")
    print(f"  Post-pool  shape={post_np.shape}  mean={post_np.mean():.4f}  "
          f"std={post_np.std():.4f}  sparsity={(post_np==0).mean():.2%}")


### 4.3 Pooling Strategy Trade-offs

Each down-sampling approach has strengths and weaknesses depending on the task.


In [ ]:
# ── Pooling strategy trade-offs ───────────────────────────────────────────────
tradeoff_rows = [
    {
        "Variant":     "MaxPool",
        "Params":      "No extra params",
        "Translation invariance": "Strong (max is position-agnostic within window)",
        "Gradient":    "Sparse (flows only through the max element)",
        "Best for":    "Sharp, sparse features (edges, corners)",
    },
    {
        "Variant":     "AvgPool",
        "Params":      "No extra params",
        "Translation invariance": "Weaker (mean averages all positions equally)",
        "Gradient":    "Dense (flows through all elements uniformly)",
        "Best for":    "Smooth, distributed activations; texture patterns",
    },
    {
        "Variant":     "GAP",
        "Params":      "No extra params; eliminates large FC layers",
        "Translation invariance": "Complete (collapses entire spatial dim)",
        "Gradient":    "Dense; encourages channel specialisation",
        "Best for":    "Modern lightweight architectures (ResNet, MobileNet)",
    },
    {
        "Variant":     "Stride-2 Conv",
        "Params":      "Extra (learnable stride-2 conv layer)",
        "Translation invariance": "Learned; adapts to data distribution",
        "Gradient":    "Dense; learnable selectivity",
        "Best for":    "When down-sampling pattern should be data-driven",
    },
]

tradeoff_df = pd.DataFrame(tradeoff_rows)
print("Pooling strategy trade-offs:")
for _, row in tradeoff_df.iterrows():
    print(f"\n  {row['Variant']}:")
    print(f"    Params  : {row['Params']}")
    print(f"    Inv.    : {row['Translation invariance']}")
    print(f"    Gradient: {row['Gradient']}")
    print(f"    Best for: {row['Best for']}")

# ── RF coverage summary ───────────────────────────────────────────────────────
print("\nReceptive field summary after 5 layers (28x28 input):")
for label, rf_data in [("SimpleCNN/MaxPool", rf_simple),
                        ("Stride-2 Conv",    rf_stride2),
                        ("VGG-style",        rf_vgg)]:
    final_rf   = rf_data[-1]["RF"]
    coverage   = min(final_rf / 28 * 100, 100)
    print(f"  {label:<22s}  RF={final_rf:>4d}px  ({coverage:.0f}% of 28px input)")


# ── Accuracy vs parameter count scatter ───────────────────────────────────────
fig_sc, ax_sc = plt.subplots(figsize=(7, 4))
for ptype in pool_variants:
    h   = histories[ptype]
    m_v = h["model"]
    n_p = sum(p.numel() for p in m_v.parameters())
    ax_sc.scatter(n_p, h["test_acc"] * 100, s=120, color=colors[ptype],
                  label=ptype, zorder=5, edgecolors="k", lw=0.7)
    ax_sc.annotate(ptype, (n_p, h["test_acc"] * 100),
                   textcoords="offset points", xytext=(5, 4), fontsize=8)

ax_sc.set_xlabel("Total Parameters", fontsize=11)
ax_sc.set_ylabel("Test Accuracy (%)", fontsize=11)
ax_sc.set_title("Test Accuracy vs Parameter Count by Pooling Strategy",
                fontsize=11, fontweight="bold")
ax_sc.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ── Final RF + accuracy combined table ────────────────────────────────────────
rf_final_rows: list[dict] = []
for ptype in pool_variants:
    h   = histories[ptype]
    n_p = sum(p.numel() for p in h["model"].parameters())
    # Use SimpleCNN RF for max/avg/gap; stride-2 RF for stride2
    if ptype in ("maxpool", "avgpool", "gap"):
        final_rf = rf_simple[-1]["RF"]
    else:
        final_rf = rf_stride2[-1]["RF"]
    rf_final_rows.append({
        "Variant":    ptype,
        "Params":     n_p,
        "Final RF":   final_rf,
        "Test Acc":   f"{h['test_acc']:.4f}",
        "Train Time": f"{h['total_time']:.1f}s",
    })

final_df = pd.DataFrame(rf_final_rows)
print("Final comparison: pooling strategy, RF, params, and test accuracy:")
print(final_df.to_string(index=False))


## Part 5 — Summary & Lessons Learned

### Key Takeaways

- **MaxPool preserves peak responses; AvgPool smooths.** MaxPool keeps the dominant
  activation in each window (sparse, strong gradient through 1 element); AvgPool
  diffuses the gradient uniformly through all window elements.
- **Global Average Pool eliminates large FC layers** — replacing the FC head with
  GAP reduces parameter count substantially and makes the network spatially agnostic
  to input size.  This is the design choice in ResNet, MobileNet, and most modern CNNs.
- **Stride-2 convolution is learnable down-sampling** — it adds parameters but lets
  the network learn *which* features to retain during spatial reduction.  Modern
  architectures (e.g., ResNet, EfficientNet) increasingly favour stride-2 conv over
  MaxPool.
- **The receptive field grows multiplicatively with stride.** After each stride-2
  operation (pool or conv), the RF jump doubles — so 5 layers of stride-2 ops give
  a jump of 32, meaning each output neuron "sees" a 32-pixel step in the input.
- **All four strategies reach similar accuracy on MNIST** — the dataset is too simple
  to differentiate them; differences become meaningful on harder tasks (CIFAR-10, ImageNet).

### What's Next

→ **06-03 (LeNet-5 & AlexNet)** combines what we know about convolution (06-01) and
pooling (06-02) to build the two landmark CNN architectures and benchmark them on CIFAR-10.
